## Environment Setup and Protocol Imports

In [2]:
# ============================================================
# Cell 1 - Environment Setup and Protocol Imports
# ============================================================

import os
import sys
import time
import pickle
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn


# ------------------------------------------------------------
# Project Directory Path Resolution
# ------------------------------------------------------------

# Use an explicit project path when provided; otherwise search
# standard local and Kaggle locations for the required modules.
CODE_DIR_ENV = os.environ.get("THESIS_CODE_DIR")

if CODE_DIR_ENV:
    CODE_DIR = Path(CODE_DIR_ENV).resolve()

else:
    candidates = []

    # Check the current working directory first.
    cwd = Path.cwd()

    if (
        (cwd / "data_pipeline.py").exists()
        and (cwd / "experiment_protocol.py").exists()
    ):
        candidates.append(cwd)

    # Search Kaggle's input directory for a directory containing
    # both required project modules.
    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():
        for pipeline_file in kaggle_input.rglob("data_pipeline.py"):
            candidate_dir = pipeline_file.parent

            if (candidate_dir / "experiment_protocol.py").exists():
                candidates.append(candidate_dir)

    # Normalize paths and remove duplicate candidates.
    candidates = list(
        dict.fromkeys(
            path.resolve()
            for path in candidates
        )
    )

    if len(candidates) == 1:
        CODE_DIR = candidates[0]

    elif len(candidates) == 0:
        raise FileNotFoundError(
            "Could not locate data_pipeline.py and "
            "experiment_protocol.py in the same directory.\n\n"
            "Set THESIS_CODE_DIR explicitly if necessary."
        )

    else:
        # Ambiguous discovery is treated as an error to avoid
        # importing project code from an unintended location.
        raise RuntimeError(
            "Multiple valid code directories were found:\n\n"
            + "\n".join(str(path) for path in candidates)
            + "\n\n"
            "Set THESIS_CODE_DIR explicitly to the "
            "directory you want to use."
        )


# ------------------------------------------------------------
# System Path Injection
# ------------------------------------------------------------

# Add the resolved project directory to the front of sys.path
# so the notebook imports the intended local project modules.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


# ------------------------------------------------------------
# Import Project Modules
# ------------------------------------------------------------

from data_pipeline import load_prepared_experiment

# Import the centralized protocol configuration used to keep
# data preparation and model evaluation consistent.
from experiment_protocol import (
    PROTOCOL_VERSION,
    EXPECTED_DATASETS,
    TRAINING_SIZES,
    SUBSET_SEEDS,
    TEST_SIZE,
    TEST_SEED,
    MODEL_SEED,
    CLASSIFICATION_THRESHOLD,
    PRIMARY_METRICS,
    SECONDARY_METRICS,
    TABPFN_MODEL_VERSION,
)


# ------------------------------------------------------------
# Evaluation Metrics
# ------------------------------------------------------------

# Import the metrics used by the standardized evaluation protocol.
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


# ------------------------------------------------------------
# Runtime Diagnostics
# ------------------------------------------------------------

# Print the active software environment and protocol settings
# to make the experiment configuration easy to audit.
print("TabPFN baseline — revised protocol")
print("=" * 60)

print(f"Python:            {sys.version.split()[0]}")
print(f"NumPy:             {np.__version__}")
print(f"Pandas:             {pd.__version__}")
print(f"scikit-learn:      {sklearn.__version__}")

print(f"Protocol:          {PROTOCOL_VERSION}")
print(f"TabPFN model:      {TABPFN_MODEL_VERSION}")

print(f"Project code dir:  {CODE_DIR}")
print(f"Datasets:          {EXPECTED_DATASETS}")
print(f"Training sizes:    {TRAINING_SIZES}")
print(f"Subset seeds:      {SUBSET_SEEDS}")
print(f"Test size:         {TEST_SIZE}")
print(f"Test seed:         {TEST_SEED}")
print(f"Model seed:        {MODEL_SEED}")
print(f"Threshold:         {CLASSIFICATION_THRESHOLD}")

print("=" * 60)
print("✓ Project modules loaded successfully.")

TabPFN baseline — revised protocol
Python:            3.12.13
NumPy:             2.0.2
Pandas:             2.3.3
scikit-learn:      1.6.1
Protocol:          low_data_tabpfn_v1
TabPFN model:      V3
Project code dir:  /kaggle/input/datasets/nicolausdeemese/iu-thesis-nicolas/files
Datasets:          ('diabetes', 'cardiovascular')
Training sizes:    (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
Subset seeds:      (42, 123, 456, 789, 1011)
Test size:         0.2
Test seed:         42
Model seed:        42
Threshold:         0.5
✓ Project modules loaded successfully.


## Install and Verify TabPFN

In [3]:
# ============================================================
# Cell 2 - Install and Verify TabPFN
# ============================================================

# Pin the package version used by the validated TabPFN-V3 experiment.
# The model version is controlled separately by TABPFN_MODEL_VERSION
# in the centralized experiment protocol.
TABPFN_PACKAGE_VERSION = "8.3.0"

print(
    f"Installing TabPFN package version "
    f"{TABPFN_PACKAGE_VERSION}..."
)
print("-" * 60)

# Install through the active Python interpreter to ensure that
# the package is added to the same environment as the notebook.
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        f"tabpfn=={TABPFN_PACKAGE_VERSION}",
    ],
    capture_output=True,
    text=True,
)

# Stop immediately if pip reports an installation failure.
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("TabPFN installation failed.")

print("✓ pip installation completed.")
print()


# ------------------------------------------------------------
# Import TabPFN
# ------------------------------------------------------------

import tabpfn
from tabpfn import TabPFNClassifier

# Read the installed package version and compare it with
# the exact version required by the experiment.
installed_version = tabpfn.__version__

print(f"Installed TabPFN version: {installed_version}")

assert installed_version == TABPFN_PACKAGE_VERSION, (
    f"Expected TabPFN {TABPFN_PACKAGE_VERSION}, "
    f"but installed {installed_version}."
)

print("✓ TabPFNClassifier imported successfully.")
print("✓ TabPFN package version is pinned and verified.")

print()
print("=" * 60)
print("TABPFN PACKAGE VERIFIED")
print("=" * 60)

Installing TabPFN package version 8.3.0...
------------------------------------------------------------
✓ pip installation completed.

Installed TabPFN version: 8.3.0
✓ TabPFNClassifier imported successfully.
✓ TabPFN package version is pinned and verified.

TABPFN PACKAGE VERIFIED


## Verify TabPFN V3 API

In [4]:
# ============================================================
# Cell 3 — Verify TabPFN V3 API
# ============================================================

import inspect

from tabpfn.constants import ModelVersion

print("TabPFN API verification")
print("=" * 60)

print(f"TabPFN package version: {tabpfn.__version__}")
print(f"Protocol model version: {TABPFN_MODEL_VERSION}")
print()


# ------------------------------------------------------------
# Verify protocol model selection
# ------------------------------------------------------------

# Ensure this notebook is explicitly configured for TabPFN V3.
assert TABPFN_MODEL_VERSION == "V3", (
    "Notebook 04 is designed for TabPFN V3. "
    f"Found protocol value: {TABPFN_MODEL_VERSION!r}"
)

# Confirm that the installed package exposes the required V3 enum.
assert hasattr(ModelVersion, "V3"), (
    "Installed TabPFN package does not expose ModelVersion.V3."
)

TABPFN_VERSION_ENUM = ModelVersion.V3

print("✓ ModelVersion.V3 is available.")
print("✓ Protocol explicitly selects TabPFN V3.")
print()


# ------------------------------------------------------------
# Verify classifier API
# ------------------------------------------------------------

# Confirm that the imported classifier is available before
# checking the methods required by the experiment.
assert TabPFNClassifier is not None

required_methods = [
    "fit",
    "predict",
    "predict_proba",
    "get_inference_config",
    "create_default_for_version",
]

# Check the complete required API before running any experiments.
missing_methods = [
    method
    for method in required_methods
    if not hasattr(TabPFNClassifier, method)
]

assert not missing_methods, (
    f"Missing required TabPFNClassifier methods: {missing_methods}"
)

for method in required_methods:
    print(f"✓ {method}() available.")

print()


# ------------------------------------------------------------
# Inspect model constructor
# ------------------------------------------------------------

# Inspect the constructor dynamically so the notebook can verify
# that the installed package supports the intended configuration.
signature = inspect.signature(TabPFNClassifier)

print("TabPFNClassifier constructor:")
print(signature)
print()

required_parameters = [
    "device",
    "random_state",
    "n_estimators",
    "show_progress_bar",
]

# Verify that all parameters required by the planned experiment
# are present in the installed classifier API.
missing_parameters = [
    parameter
    for parameter in required_parameters
    if parameter not in signature.parameters
]

assert not missing_parameters, (
    f"Missing constructor parameters: {missing_parameters}"
)

print("Required constructor parameters verified:")
for parameter in required_parameters:
    print(f"  ✓ {parameter}")

print()
print("=" * 60)
print("TABPFN V3 API VERIFICATION PASSED")
print("=" * 60)

TabPFN API verification
TabPFN package version: 8.3.0
Protocol model version: V3

✓ ModelVersion.V3 is available.
✓ Protocol explicitly selects TabPFN V3.

✓ fit() available.
✓ predict() available.
✓ predict_proba() available.
✓ get_inference_config() available.
✓ create_default_for_version() available.

TabPFNClassifier constructor:
(*, n_estimators: "int | Literal['auto']" = 'auto', auto_scale_n_estimators: 'bool' = True, categorical_features_indices: 'Sequence[int] | None' = None, softmax_temperature: 'float' = 0.9, balance_probabilities: 'bool' = False, average_before_softmax: 'bool' = False, model_path: "str | Path | list[str] | list[Path] | Literal['auto'] | ClassifierModelSpecs | list[ClassifierModelSpecs]" = 'auto', device: 'DevicesSpecification' = 'auto', ignore_pretraining_limits: 'bool' = False, inference_precision: "_dtype | Literal['autocast', 'auto']" = 'auto', fit_mode: "Literal['low_memory', 'fit_preprocessors', 'fit_with_cache', 'batched']" = 'fit_preprocessors', memor

## Configure TabPFN Authentication

In [5]:
# ============================================================
# Cell 4 — Configure TabPFN Authentication
# ============================================================

from kaggle_secrets import UserSecretsClient

print("Configuring TabPFN authentication...")
print("-" * 60)

# Access secrets through Kaggle's secure secrets interface
# rather than hard-coding credentials in the notebook.
user_secrets = UserSecretsClient()

TABPFN_TOKEN = user_secrets.get_secret("TABPFN_TOKEN")

# Fail early if the expected secret is unavailable or empty.
assert TABPFN_TOKEN is not None, (
    "TABPFN_TOKEN secret was not retrieved."
)

assert len(TABPFN_TOKEN.strip()) > 0, (
    "TABPFN_TOKEN secret is empty."
)

# Expose the token to TabPFN through the environment without
# printing or otherwise exposing its value in notebook output.
os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN

print("✓ TABPFN_TOKEN retrieved from Kaggle Secrets.")
print("✓ TABPFN_TOKEN configured in environment.")

print()
print("=" * 60)
print("TABPFN AUTHENTICATION CONFIGURED")
print("=" * 60)

Configuring TabPFN authentication...
------------------------------------------------------------
✓ TABPFN_TOKEN retrieved from Kaggle Secrets.
✓ TABPFN_TOKEN configured in environment.

TABPFN AUTHENTICATION CONFIGURED


## Load Prepared Experimental Data

In [6]:
# ============================================================
# Cell 5 - Load Prepared Experimental Data
# ============================================================

print("Loading prepared experimental data...")
print("-" * 60)

# Load the fixed datasets, splits, training subsets, and metadata
# generated by the centralized data preparation pipeline.
experiment_data = load_prepared_experiment()

print("✓ Prepared experimental data loaded successfully.")
print()
print(f"Object type:    {type(experiment_data)}")
print(f"Top-level keys: {list(experiment_data.keys())}")

Loading prepared experimental data...
------------------------------------------------------------
✓ Prepared experimental data loaded successfully.

Object type:    <class 'dict'>
Top-level keys: ['prepared_data', 'split_data', 'training_subsets', 'config', 'manifest']


## Verify and Extract Experiment Data

In [7]:
# ============================================================
# Cell 6 - Verify and Extract Experiment Data
# ============================================================

# Define the exact top-level components expected from the
# prepared experiment loader.
EXPECTED_KEYS = {
    "prepared_data",
    "split_data",
    "training_subsets",
    "config",
    "manifest",
}

actual_keys = set(experiment_data.keys())

print("Verifying prepared experimental data structure...")
print("-" * 60)

# Ensure that no expected component is missing and no unexpected
# component has been introduced into the prepared experiment object.
assert actual_keys == EXPECTED_KEYS, (
    f"Unexpected experiment data structure.\n"
    f"Expected: {EXPECTED_KEYS}\n"
    f"Found:    {actual_keys}"
)

# Extract the prepared datasets, fixed test splits, training
# subsets, protocol configuration, and preparation manifest.
prepared_data = experiment_data["prepared_data"]
split_data = experiment_data["split_data"]
training_subsets = experiment_data["training_subsets"]
config = experiment_data["config"]
manifest = experiment_data["manifest"]

print("✓ Top-level structure verified.")
print()
print("Extracted components:")
print(f"  ✓ prepared_data:    {type(prepared_data)}")
print(f"  ✓ split_data:       {type(split_data)}")
print(f"  ✓ training_subsets: {type(training_subsets)}")
print(f"  ✓ config:           {type(config)}")
print(f"  ✓ manifest:         {type(manifest)}")

Verifying prepared experimental data structure...
------------------------------------------------------------
✓ Top-level structure verified.

Extracted components:
  ✓ prepared_data:    <class 'dict'>
  ✓ split_data:       <class 'dict'>
  ✓ training_subsets: <class 'dict'>
  ✓ config:           <class 'dict'>
  ✓ manifest:         <class 'dict'>


## Verify Experimental Configuration

In [8]:
# ============================================================
# Cell 7 - Verify Experimental Configuration
# ============================================================

print("Experimental configuration verification")
print("=" * 60)

# Extract dataset names from the manifest while preserving
# their stored order for comparison with the central protocol.
manifest_datasets = tuple(manifest["datasets"].keys())

print(f"Datasets:          {EXPECTED_DATASETS}")
print(f"Training sizes:    {TRAINING_SIZES}")
print(f"Subset seeds:      {SUBSET_SEEDS}")
print(f"Test size:         {config['test_size']}")
print(f"Test seed:         {config['test_seed']}")
print(f"Number of classes: {config['n_classes']}")
print()


# ------------------------------------------------------------
# Compare manifest configuration with central protocol
# ------------------------------------------------------------

# Verify that all training subset sizes and subset seeds were
# generated according to the centralized experimental protocol.
assert tuple(config["training_sizes"]) == TRAINING_SIZES, (
    "Manifest training sizes do not match the central protocol."
)

assert tuple(config["training_seeds"]) == SUBSET_SEEDS, (
    "Manifest training seeds do not match the central protocol."
)

# Verify the fixed test split configuration.
assert config["test_size"] == TEST_SIZE, (
    "Manifest test size does not match the central protocol."
)

assert config["test_seed"] == TEST_SEED, (
    "Manifest test seed does not match the central protocol."
)

# This notebook is restricted to binary classification.
assert config["n_classes"] == 2, (
    "Notebook 04 expects binary classification."
)

# Verify that the prepared experiment contains exactly the
# datasets defined by the central protocol.
assert manifest_datasets == EXPECTED_DATASETS, (
    f"Manifest datasets do not match the central protocol.\n"
    f"Expected: {EXPECTED_DATASETS}\n"
    f"Found:    {manifest_datasets}"
)

# Verify that all major experiment components use the same
# dataset set and ordering as the protocol.
assert tuple(training_subsets.keys()) == EXPECTED_DATASETS
assert tuple(split_data.keys()) == EXPECTED_DATASETS
assert tuple(prepared_data.keys()) == EXPECTED_DATASETS

print("✓ Dataset configuration verified.")
print("✓ Training sizes verified.")
print("✓ Subset seeds verified.")
print("✓ Test size verified.")
print("✓ Test seed verified.")
print("✓ Binary classification verified.")
print("✓ Dataset ordering verified.")

print()
print("=" * 60)
print("EXPERIMENTAL CONFIGURATION VERIFIED")
print("=" * 60)

Experimental configuration verification
Datasets:          ('diabetes', 'cardiovascular')
Training sizes:    (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
Subset seeds:      (42, 123, 456, 789, 1011)
Test size:         0.2
Test seed:         42
Number of classes: 2

✓ Dataset configuration verified.
✓ Training sizes verified.
✓ Subset seeds verified.
✓ Test size verified.
✓ Test seed verified.
✓ Binary classification verified.
✓ Dataset ordering verified.

EXPERIMENTAL CONFIGURATION VERIFIED


## Verify Training Subsets and Fixed Test Sets

In [9]:
# ============================================================
# Cell 8 — Verify Training Subsets and Fixed Test Sets
# ============================================================

print("Verifying training subsets and fixed test sets...")
print("=" * 60)

for dataset_name in EXPECTED_DATASETS:

    print()
    print(f"Dataset: {dataset_name}")
    print("-" * 60)

    X_test = split_data[dataset_name]["X_test"]
    y_test = split_data[dataset_name]["y_test"]

    # Convert stored indices to pandas Index objects for reliable
    # uniqueness and set-based overlap checks.
    train_indices = pd.Index(
        split_data[dataset_name]["train_indices"]
    )
    test_indices = pd.Index(
        split_data[dataset_name]["test_indices"]
    )

    print(f"Test samples:  {len(X_test):,}")
    print(f"Test features: {X_test.shape[1]:,}")

    # --------------------------------------------------------
    # Fixed split integrity
    # --------------------------------------------------------

    # Verify feature/target alignment and absence of missing values
    # in the fixed test set used by every model configuration.
    assert len(X_test) == len(y_test)

    assert not X_test.isnull().any().any(), (
        f"Missing values in X_test for {dataset_name}."
    )

    assert not y_test.isnull().any(), (
        f"Missing values in y_test for {dataset_name}."
    )

    # Ensure the stored train/test indices are unique and disjoint.
    assert train_indices.is_unique
    assert test_indices.is_unique

    assert set(train_indices).isdisjoint(set(test_indices)), (
        f"Train/test overlap detected in {dataset_name}."
    )

    # --------------------------------------------------------
    # Training subset integrity
    # --------------------------------------------------------

    for training_size in TRAINING_SIZES:

        # Confirm that every protocol-defined training size exists
        # for the current dataset.
        assert training_size in training_subsets[dataset_name]

        for subset_seed in SUBSET_SEEDS:

            # Confirm that every required subset seed is available
            # for each training size.
            assert subset_seed in training_subsets[
                dataset_name
            ][training_size]

            subset = training_subsets[
                dataset_name
            ][training_size][subset_seed]

            X_train = subset["X"]
            y_train = subset["y"]
            subset_indices = pd.Index(subset["indices"])

            # Verify that the subset contains exactly the number
            # of observations specified by the protocol.
            assert len(X_train) == training_size, (
                f"Incorrect training size for "
                f"{dataset_name}, n={training_size}, "
                f"subset_seed={subset_seed}."
            )

            assert len(y_train) == training_size

            # Ensure training and test matrices use the same
            # feature dimensionality.
            assert X_train.shape[1] == X_test.shape[1]

            # Each stored subset index must be unique and must come
            # exclusively from the fixed training split.
            assert subset_indices.is_unique

            assert set(subset_indices).issubset(
                set(train_indices)
            ), (
                f"Subset contains rows outside the fixed "
                f"training split for {dataset_name}, "
                f"n={training_size}, subset_seed={subset_seed}."
            )

            # Explicitly verify that no subset row appears in the
            # held-out test split.
            assert set(subset_indices).isdisjoint(
                set(test_indices)
            ), (
                f"Training/test leakage detected for "
                f"{dataset_name}, n={training_size}, "
                f"subset_seed={subset_seed}."
            )

            # Verify that the prepared subset contains no missing
            # feature values or target labels.
            assert not X_train.isnull().any().any()
            assert not y_train.isnull().any()

            # ----------------------------------------------------
            # Binary class balance
            # ----------------------------------------------------

            # The protocol requires equal numbers of class 0 and
            # class 1 observations in every training subset.
            class_counts = y_train.value_counts().to_dict()

            assert set(class_counts.keys()) == {0, 1}, (
                f"Unexpected classes in "
                f"{dataset_name}, n={training_size}, "
                f"subset_seed={subset_seed}: {class_counts}"
            )

            assert class_counts[0] == training_size // 2
            assert class_counts[1] == training_size // 2

    print("✓ Fixed test split verified.")
    print(f"✓ All {len(TRAINING_SIZES)} training sizes verified.")
    print(f"✓ All {len(SUBSET_SEEDS)} subset seeds verified.")
    print("✓ Binary balance verified for every subset.")

print()
print("=" * 60)
print("✓ ALL EXPERIMENTAL DATA SUBSETS VERIFIED")
print("=" * 60)

Verifying training subsets and fixed test sets...

Dataset: diabetes
------------------------------------------------------------
Test samples:  45,895
Test features: 21
✓ Fixed test split verified.
✓ All 11 training sizes verified.
✓ All 5 subset seeds verified.
✓ Binary balance verified for every subset.

Dataset: cardiovascular
------------------------------------------------------------
Test samples:  13,723
Test features: 11
✓ Fixed test split verified.
✓ All 11 training sizes verified.
✓ All 5 subset seeds verified.
✓ Binary balance verified for every subset.

✓ ALL EXPERIMENTAL DATA SUBSETS VERIFIED


## Define Evaluation Function

In [10]:
# ============================================================
# Cell 9 — Define Evaluation Function
# ============================================================

# Combine the primary and secondary metrics defined by the
# centralized thesis protocol for completeness checks.
ALL_METRICS = PRIMARY_METRICS + SECONDARY_METRICS

print("Evaluation protocol")
print("=" * 60)

print("Primary metrics:")
for metric in PRIMARY_METRICS:
    print(f"  - {metric}")

print()
print("Secondary metrics:")
for metric in SECONDARY_METRICS:
    print(f"  - {metric}")

print()
print(
    f"Classification threshold: "
    f"{CLASSIFICATION_THRESHOLD}"
)


def evaluate_predictions(y_true, y_score):
    """
    Evaluate binary predictions using the central thesis protocol.

    Parameters
    ----------
    y_true : array-like
        Ground-truth binary labels.

    y_score : array-like
        Predicted probability or score for the positive class.
        Threshold-free metrics use these values directly.

    Returns
    -------
    dict
        Evaluation metrics defined by the thesis protocol.
    """

    # Convert inputs to NumPy arrays for consistent metric handling.
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # This function expects one score per observation.
    assert y_score.ndim == 1
    assert len(y_true) == len(y_score)

    # Convert continuous positive-class scores into binary labels
    # using the fixed threshold defined by the central protocol.
    y_pred = (
        y_score >= CLASSIFICATION_THRESHOLD
    ).astype(int)

    return {
        # ROC-AUC and PR-AUC use the continuous positive-class
        # scores and are therefore independent of the threshold.
        "roc_auc": float(
            roc_auc_score(y_true, y_score)
        ),
        "pr_auc": float(
            average_precision_score(y_true, y_score)
        ),

        # The remaining metrics evaluate thresholded predictions.
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
    }


print()
print("✓ Evaluation function defined.")
print("=" * 60)

Evaluation protocol
Primary metrics:
  - roc_auc
  - pr_auc

Secondary metrics:
  - accuracy
  - balanced_accuracy
  - precision
  - recall
  - f1

Classification threshold: 0.5

✓ Evaluation function defined.


## Define TabPFN V3 Model Factory

In [11]:
# ============================================================
# Cell 10 — Define TabPFN V3 Model Factory
# ============================================================

import torch

print("Defining TabPFN V3 model factory...")
print("-" * 60)


def create_tabpfn_model():
    """
    Create a fresh TabPFN V3 classifier.

    Seed separation:
      - SUBSET_SEEDS select the predefined training subsets.
      - MODEL_SEED controls model-side randomness.
    """

    # Create a new classifier for each experimental run so that
    # model state is never shared between configurations.
    return TabPFNClassifier.create_default_for_version(
        TABPFN_VERSION_ENUM,
        device="auto",
        random_state=MODEL_SEED,
        n_estimators="auto",
        show_progress_bar=False,
    )


# Instantiate one model to verify that the factory applies the
# required reproducibility and estimator settings.
test_model = create_tabpfn_model()

assert test_model.random_state == MODEL_SEED
assert test_model.n_estimators == "auto"

# Report the hardware PyTorch can see; device="auto" chooses from it.
if torch.cuda.is_available():
    visible_device = (
        f"cuda ({torch.cuda.device_count()} x {torch.cuda.get_device_name(0)})"
    )
else:
    visible_device = "cpu"

print("✓ TabPFN V3 factory defined.")
print(f"✓ Model seed fixed at: {MODEL_SEED}")
print(f"✓ Device setting: auto | PyTorch sees: {visible_device}")
print("✓ n_estimators: auto")

Defining TabPFN V3 model factory...
------------------------------------------------------------
✓ TabPFN V3 factory defined.
✓ Model seed fixed at: 42
✓ Device setting: auto | PyTorch sees: cuda (2 x Tesla T4)
✓ n_estimators: auto


## Verify TabPFN V3 Weight Loading

In [12]:
# ============================================================
# Cell 11 — Verify TabPFN V3 Weight Loading
# ============================================================

print("Testing TabPFN V3 model initialization...")
print("=" * 60)

try:
    # Create a fresh model instance to test initialization independently
    # of the models used in the actual experimental runs.
    weight_test_model = create_tabpfn_model()

    # Public API: obtain the inference configuration and trigger the
    # required pretrained checkpoint loading without fitting on data.
    inference_config = weight_test_model.get_inference_config()

    assert inference_config is not None

    print("✓ TabPFN V3 classifier created.")
    print("✓ TabPFN V3 checkpoint loaded successfully.")

except Exception as exc:
    # Report the initialization error while preserving the original
    # exception for debugging and preventing invalid experiments.
    print()
    print("✗ TabPFN V3 model initialization failed.")
    print(f"Error type: {type(exc).__name__}")
    print(f"Error: {exc}")
    raise

print()
print("=" * 60)
print("TABPFN V3 WEIGHT-LOADING TEST PASSED")
print("=" * 60)

Testing TabPFN V3 model initialization...


tabpfn-v3-classifier-v3_default.ckpt:   0%|          | 0.00/213M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

✓ TabPFN V3 classifier created.
✓ TabPFN V3 checkpoint loaded successfully.

TABPFN V3 WEIGHT-LOADING TEST PASSED


## Run TabPFN V3 Smoke Test

In [13]:
# ============================================================
# Cell 12 — Run TabPFN V3 Smoke Test
# ============================================================

# Use one predefined training subset and a small portion of the
# fixed test set to validate the complete experimental pipeline.
SMOKE_DATASET = EXPECTED_DATASETS[0]
SMOKE_TRAINING_SIZE = 128
SMOKE_SUBSET_SEED = SUBSET_SEEDS[0]
SMOKE_TEST_SIZE = 1000

print("Running TabPFN V3 smoke test...")
print("=" * 60)

print(f"Dataset:         {SMOKE_DATASET}")
print(f"Training size:   {SMOKE_TRAINING_SIZE}")
print(f"Subset seed:     {SMOKE_SUBSET_SEED}")
print(f"Model seed:      {MODEL_SEED}")
print(f"Smoke test rows: {SMOKE_TEST_SIZE}")

print("=" * 60)

smoke_start = time.time()


# ------------------------------------------------------------
# Load predefined training subset
# ------------------------------------------------------------

# Reuse the fixed subset generated by the central data pipeline;
# no new sampling is performed during the smoke test.
smoke_subset = training_subsets[
    SMOKE_DATASET
][SMOKE_TRAINING_SIZE][SMOKE_SUBSET_SEED]

X_train = smoke_subset["X"]
y_train = smoke_subset["y"]


# ------------------------------------------------------------
# Load a small portion of the fixed test set
# ------------------------------------------------------------

# Use the beginning of the fixed test set only to reduce smoke-test
# runtime. The full fixed test set is used in the final experiment.
full_X_test = split_data[
    SMOKE_DATASET
]["X_test"]

full_y_test = split_data[
    SMOKE_DATASET
]["y_test"]

assert SMOKE_TEST_SIZE <= len(full_X_test), (
    "Smoke-test size exceeds the available test set."
)

X_test = full_X_test.iloc[:SMOKE_TEST_SIZE].copy()
y_test = full_y_test.iloc[:SMOKE_TEST_SIZE].copy()


# ------------------------------------------------------------
# Basic data validation
# ------------------------------------------------------------

assert len(X_train) == SMOKE_TRAINING_SIZE
assert len(y_train) == SMOKE_TRAINING_SIZE

assert len(X_test) == SMOKE_TEST_SIZE
assert len(y_test) == SMOKE_TEST_SIZE

# Confirm that both training and smoke-test targets contain
# both classes and that the feature dimensions are compatible.
assert set(np.asarray(y_train)) == {0, 1}
assert set(np.asarray(y_test)) == {0, 1}

assert X_train.shape[1] == X_test.shape[1]


# ------------------------------------------------------------
# Create TabPFN V3 model
# ------------------------------------------------------------

# Create a fresh model instance using the fixed model seed.
smoke_model = create_tabpfn_model()


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

training_start = time.time()

smoke_model.fit(
    X_train,
    y_train,
)

training_time = time.time() - training_start


# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

prediction_start = time.time()

y_proba = smoke_model.predict_proba(
    X_test
)

prediction_time = time.time() - prediction_start


# ------------------------------------------------------------
# Extract positive-class probability
# ------------------------------------------------------------

# Do not assume that column 1 always represents the positive class.
# Identify the probability column corresponding explicitly to class 1.
classes = np.asarray(
    smoke_model.classes_
)

assert len(classes) == 2, (
    f"Expected binary classes, found {classes}."
)

positive_indices = np.where(
    classes == 1
)[0]

assert len(positive_indices) == 1, (
    f"Positive class 1 not found in {classes}."
)

positive_index = positive_indices[0]

y_score = np.asarray(
    y_proba[:, positive_index]
)

assert len(y_score) == len(y_test), (
    "Prediction count does not match test-set size."
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

# Evaluate positive-class scores using the centralized metric
# definitions and classification threshold.
smoke_metrics = evaluate_predictions(
    y_test,
    y_score,
)


# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

# Store the smoke-test result using the same schema as the
# final experiment, while keeping it separate from final results.
smoke_result = {
    "protocol_version": PROTOCOL_VERSION,
    "dataset": SMOKE_DATASET,
    "model": "tabpfn_v3",
    "training_size": SMOKE_TRAINING_SIZE,
    "subset_seed": SMOKE_SUBSET_SEED,
    "model_seed": MODEL_SEED,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "training_time": float(training_time),
    "prediction_time": float(prediction_time),
    **smoke_metrics,
}

smoke_elapsed = time.time() - smoke_start


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print()
print("Smoke test completed.")
print("-" * 60)

for metric in PRIMARY_METRICS + SECONDARY_METRICS:
    print(
        f"{metric:20s}: "
        f"{smoke_result[metric]:.4f}"
    )

print()
print(
    f"Training time:    "
    f"{smoke_result['training_time']:.2f} s"
)

print(
    f"Prediction time:  "
    f"{smoke_result['prediction_time']:.2f} s"
)

print(
    f"Total time:       "
    f"{smoke_elapsed:.2f} s"
)


# ------------------------------------------------------------
# Smoke-result validation
# ------------------------------------------------------------

# Verify that the recorded metadata matches the active protocol.
assert smoke_result["protocol_version"] == PROTOCOL_VERSION
assert smoke_result["dataset"] == SMOKE_DATASET
assert smoke_result["model"] == "tabpfn_v3"

assert (
    smoke_result["training_size"]
    == SMOKE_TRAINING_SIZE
)

assert (
    smoke_result["subset_seed"]
    == SMOKE_SUBSET_SEED
)

assert (
    smoke_result["model_seed"]
    == MODEL_SEED
)

assert (
    smoke_result["classification_threshold"]
    == CLASSIFICATION_THRESHOLD
)

# Verify that all recorded metrics and timings are finite and
# lie within the expected range for this binary classification task.
for metric in ALL_METRICS:

    value = smoke_result[metric]

    assert np.isfinite(value), (
        f"Non-finite metric: "
        f"{metric}={value}"
    )

    assert 0.0 <= value <= 1.0, (
        f"Invalid metric: "
        f"{metric}={value}"
    )

assert smoke_result["training_time"] >= 0
assert smoke_result["prediction_time"] >= 0


print()
print("=" * 60)
print("✓ TABPFN V3 SMOKE TEST PASSED")
print("=" * 60)

Running TabPFN V3 smoke test...
Dataset:         diabetes
Training size:   128
Subset seed:     42
Model seed:      42
Smoke test rows: 1000

Smoke test completed.
------------------------------------------------------------
roc_auc             : 0.7604
pr_auc              : 0.2910
accuracy            : 0.6530
balanced_accuracy   : 0.7003
precision           : 0.2635
recall              : 0.7671
f1                  : 0.3923

Training time:    1.30 s
Prediction time:  2.18 s
Total time:       3.50 s

✓ TABPFN V3 SMOKE TEST PASSED


## Run Full TabPFN V3 Experiment

In [14]:
# ============================================================
# Cell 13 — Run Full TabPFN V3 Experiment
# ============================================================

CHECKPOINT_PATH = Path(
    "/kaggle/working/"
    "tabpfn_v3_revised_protocol_checkpoint.pkl"
)

EXPECTED_RUNS = (
    len(EXPECTED_DATASETS)
    * len(TRAINING_SIZES)
    * len(SUBSET_SEEDS)
)

# Batching affects only prediction execution; the full test set
# remains unchanged and is evaluated for every experimental run.
PREDICTION_BATCH_SIZE = 1000


print("Full TabPFN V3 experiment")
print("=" * 60)

print(f"Datasets:            {len(EXPECTED_DATASETS)}")
print(f"Training sizes:      {len(TRAINING_SIZES)}")
print(f"Subset seeds:        {len(SUBSET_SEEDS)}")
print(f"Model:               TabPFN-{TABPFN_MODEL_VERSION[1:]}")
print(f"Model seed:          {MODEL_SEED}")
print(f"Prediction batch:    {PREDICTION_BATCH_SIZE}")
print(f"Expected runs:       {EXPECTED_RUNS}")
print(f"Checkpoint:          {CHECKPOINT_PATH}")

print("=" * 60)


# ------------------------------------------------------------
# Full experiment function
# ------------------------------------------------------------

def run_tabpfn_experiment(
    dataset_name,
    training_size,
    subset_seed,
):
    """
    Run one TabPFN V3 experiment.

    Training uses one predefined balanced subset.
    Evaluation uses the complete fixed test set.
    Prediction is batched only to control memory usage.
    """

    # Validate that the requested configuration belongs
    # to the predefined experimental protocol.
    if dataset_name not in EXPECTED_DATASETS:
        raise ValueError(
            f"Unknown dataset: {dataset_name}"
        )

    if training_size not in TRAINING_SIZES:
        raise ValueError(
            f"Unknown training size: {training_size}"
        )

    if subset_seed not in SUBSET_SEEDS:
        raise ValueError(
            f"Unknown subset seed: {subset_seed}"
        )


    # --------------------------------------------------------
    # Load predefined training subset
    # --------------------------------------------------------

    subset = training_subsets[
        dataset_name
    ][training_size][subset_seed]

    X_train = subset["X"]
    y_train = subset["y"]


    # --------------------------------------------------------
    # Load complete fixed test set
    # --------------------------------------------------------

    X_test = split_data[
        dataset_name
    ]["X_test"]

    y_test = split_data[
        dataset_name
    ]["y_test"]

    assert len(X_test) == len(y_test)


    # --------------------------------------------------------
    # Validate labels and feature dimensions
    # --------------------------------------------------------

    assert set(np.asarray(y_train)) == {0, 1}
    assert set(np.asarray(y_test)) == {0, 1}

    assert X_train.shape[1] == X_test.shape[1]


    # --------------------------------------------------------
    # Create a fresh TabPFN V3 model
    # --------------------------------------------------------

    model = create_tabpfn_model()


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    training_start = time.time()

    model.fit(
        X_train,
        y_train,
    )

    training_time = (
        time.time() - training_start
    )


    # --------------------------------------------------------
    # Batched prediction on the complete test set
    # --------------------------------------------------------

    prediction_start = time.time()

    n_test = len(X_test)

    n_batches = (
        (n_test + PREDICTION_BATCH_SIZE - 1)
        // PREDICTION_BATCH_SIZE
    )

    print(
        f"  Test rows: {n_test:,} | "
        f"Prediction batches: {n_batches}"
    )

    probability_batches = []

    for batch_number, start in enumerate(
        range(
            0,
            n_test,
            PREDICTION_BATCH_SIZE,
        ),
        start=1,
    ):

        stop = min(
            start + PREDICTION_BATCH_SIZE,
            n_test,
        )

        X_batch = X_test.iloc[
            start:stop
        ]

        y_proba_batch = model.predict_proba(
            X_batch
        )

        probability_batches.append(
            np.asarray(y_proba_batch)
        )

        elapsed_prediction = (
            time.time() - prediction_start
        )

        print(
            f"    Batch "
            f"{batch_number}/{n_batches} "
            f"({stop:,}/{n_test:,} rows) | "
            f"{elapsed_prediction:.1f} s"
        )

    # Reconstruct predictions in the original test-set order.
    y_proba = np.vstack(
        probability_batches
    )

    prediction_time = (
        time.time() - prediction_start
    )


    # --------------------------------------------------------
    # Extract positive-class probability
    # --------------------------------------------------------

    classes = np.asarray(
        model.classes_
    )

    assert len(classes) == 2, (
        f"Expected binary classes, "
        f"found {classes}."
    )

    positive_indices = np.where(
        classes == 1
    )[0]

    assert len(positive_indices) == 1, (
        f"Positive class 1 not found "
        f"in {classes}."
    )

    positive_index = positive_indices[0]

    y_score = np.asarray(
        y_proba[:, positive_index]
    )

    assert len(y_score) == len(y_test), (
        f"Prediction count mismatch: "
        f"{len(y_score)} predictions for "
        f"{len(y_test)} test rows."
    )


    # --------------------------------------------------------
    # Evaluate predictions using the central protocol
    # --------------------------------------------------------

    evaluation_metrics = evaluate_predictions(
        y_test,
        y_score,
    )


    # --------------------------------------------------------
    # Construct standardized result record
    # --------------------------------------------------------

    result = {
        "protocol_version": PROTOCOL_VERSION,
        "dataset": dataset_name,
        "model": "tabpfn_v3",
        "training_size": training_size,
        "subset_seed": subset_seed,
        "model_seed": MODEL_SEED,
        "classification_threshold": (
            CLASSIFICATION_THRESHOLD
        ),
        "training_time": float(
            training_time
        ),
        "prediction_time": float(
            prediction_time
        ),
        **evaluation_metrics,
    }

    return result


# ------------------------------------------------------------
# Checkpoint metadata
# ------------------------------------------------------------

CHECKPOINT_PROTOCOL = {
    "protocol_version": PROTOCOL_VERSION,
    "tabpfn_model_version": TABPFN_MODEL_VERSION,
    "training_sizes": tuple(
        TRAINING_SIZES
    ),
    "subset_seeds": tuple(
        SUBSET_SEEDS
    ),
    "model_seed": MODEL_SEED,
    "prediction_batch_size": (
        PREDICTION_BATCH_SIZE
    ),
}


# ------------------------------------------------------------
# Atomic checkpoint save
# ------------------------------------------------------------

def save_checkpoint(results):
    """
    Save experiment results atomically.

    A temporary file is written first and then replaced
    with the target checkpoint to reduce corruption risk.
    """

    payload = {
        "metadata": CHECKPOINT_PROTOCOL,
        "results": results,
    }

    temporary_path = CHECKPOINT_PATH.with_suffix(
        ".tmp"
    )

    with open(
        temporary_path,
        "wb",
    ) as handle:

        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    temporary_path.replace(
        CHECKPOINT_PATH
    )


# ------------------------------------------------------------
# Checkpoint loading
# ------------------------------------------------------------

def load_checkpoint():

    with open(
        CHECKPOINT_PATH,
        "rb",
    ) as handle:

        payload = pickle.load(
            handle
        )

    assert isinstance(
        payload,
        dict,
    )

    assert payload["metadata"] == (
        CHECKPOINT_PROTOCOL
    ), (
        "Checkpoint protocol metadata "
        "does not match the current experiment."
    )

    results = payload["results"]

    assert isinstance(
        results,
        list,
    )

    return results


# ------------------------------------------------------------
# Load existing checkpoint or initialize results
# ------------------------------------------------------------

if CHECKPOINT_PATH.exists():

    results = load_checkpoint()

    print(
        f"Existing revised checkpoint "
        f"loaded: {len(results)} completed runs."
    )

else:

    results = []

    print(
        "No revised checkpoint found."
    )

    print(
        "Starting experiment from the beginning."
    )


# ------------------------------------------------------------
# Identify completed runs
# ------------------------------------------------------------

completed_runs = {
    (
        r["dataset"],
        r["training_size"],
        r["subset_seed"],
    )
    for r in results
}

assert len(completed_runs) == len(results), (
    "Duplicate run keys detected in checkpoint."
)

assert all(
    r.get("protocol_version")
    == PROTOCOL_VERSION
    for r in results
), (
    "Checkpoint contains results from "
    "another protocol."
)

assert all(
    r.get("model")
    == "tabpfn_v3"
    for r in results
), (
    "Checkpoint contains non-V3 results."
)

assert all(
    r.get("model_seed")
    == MODEL_SEED
    for r in results
), (
    "Checkpoint contains a different "
    "model seed."
)

print(
    f"Completed runs detected: "
    f"{len(completed_runs)}/{EXPECTED_RUNS}"
)


# ------------------------------------------------------------
# Execute full experiment
# ------------------------------------------------------------

experiment_start = time.time()

run_counter = len(
    completed_runs
)

for dataset_name in EXPECTED_DATASETS:

    for training_size in TRAINING_SIZES:

        for subset_seed in SUBSET_SEEDS:

            run_key = (
                dataset_name,
                training_size,
                subset_seed,
            )

            # Skip configurations already stored in the checkpoint.
            if run_key in completed_runs:
                continue

            run_counter += 1

            print()
            print(
                f"[{run_counter}/{EXPECTED_RUNS}] "
                f"{dataset_name} | "
                f"TabPFN-{TABPFN_MODEL_VERSION[1:]} | "
                f"n={training_size} | "
                f"subset_seed={subset_seed}"
            )

            run_start = time.time()

            result = run_tabpfn_experiment(
                dataset_name=dataset_name,
                training_size=training_size,
                subset_seed=subset_seed,
            )

            run_elapsed = (
                time.time() - run_start
            )


            # ------------------------------------------------
            # Validate result before checkpointing
            # ------------------------------------------------

            expected_result_keys = {
                "protocol_version",
                "dataset",
                "model",
                "training_size",
                "subset_seed",
                "model_seed",
                "classification_threshold",
                "training_time",
                "prediction_time",
                *ALL_METRICS,
            }

            missing_keys = (
                expected_result_keys
                - set(result.keys())
            )

            assert not missing_keys, (
                f"Missing result keys: "
                f"{missing_keys}"
            )

            assert (
                result["protocol_version"]
                == PROTOCOL_VERSION
            )

            assert (
                result["dataset"]
                == dataset_name
            )

            assert (
                result["model"]
                == "tabpfn_v3"
            )

            assert (
                result["training_size"]
                == training_size
            )

            assert (
                result["subset_seed"]
                == subset_seed
            )

            assert (
                result["model_seed"]
                == MODEL_SEED
            )

            assert (
                result["classification_threshold"]
                == CLASSIFICATION_THRESHOLD
            )

            for metric in ALL_METRICS:

                value = result[metric]

                assert np.isfinite(value), (
                    f"Non-finite {metric}: "
                    f"{value}"
                )

                assert 0.0 <= value <= 1.0, (
                    f"Invalid {metric}: "
                    f"{value}"
                )

            assert (
                result["training_time"] >= 0
            )

            assert (
                result["prediction_time"] >= 0
            )


            # ------------------------------------------------
            # Append result and save checkpoint
            # ------------------------------------------------

            results.append(result)

            completed_runs.add(
                run_key
            )

            save_checkpoint(
                results
            )


            # ------------------------------------------------
            # Progress output
            # ------------------------------------------------

            print(
                f"  ROC-AUC: "
                f"{result['roc_auc']:.4f} | "
                f"PR-AUC: "
                f"{result['pr_auc']:.4f}"
            )

            print(
                f"  Training time: "
                f"{result['training_time']:.2f} s"
            )

            print(
                f"  Prediction time: "
                f"{result['prediction_time']:.2f} s"
            )

            print(
                f"  Run time: "
                f"{run_elapsed:.2f} s"
            )

            print(
                f"  Checkpoint saved: "
                f"{len(results)}/"
                f"{EXPECTED_RUNS}"
            )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

elapsed_time = (
    time.time() - experiment_start
)

print()
print("=" * 60)
print("Full TabPFN V3 experiment completed.")
print("=" * 60)

print(
    f"Completed runs: {len(results)}"
)

print(
    f"Expected runs:  {EXPECTED_RUNS}"
)

print(
    f"Elapsed time:   "
    f"{elapsed_time:.2f} seconds"
)

assert len(results) == EXPECTED_RUNS

assert (
    len(completed_runs)
    == EXPECTED_RUNS
)

print(
    f"✓ All {EXPECTED_RUNS} "
    f"TabPFN V3 runs completed."
)

print(
    f"✓ Revised checkpoint: "
    f"{CHECKPOINT_PATH}"
)

Full TabPFN V3 experiment
Datasets:            2
Training sizes:      11
Subset seeds:        5
Model:               TabPFN-3
Model seed:          42
Prediction batch:    1000
Expected runs:       110
Checkpoint:          /kaggle/working/tabpfn_v3_revised_protocol_checkpoint.pkl
No revised checkpoint found.
Starting experiment from the beginning.
Completed runs detected: 0/110

[1/110] diabetes | TabPFN-3 | n=8 | subset_seed=42


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as a different type depending on the ColumnTransformer inputs.
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


  Test rows: 45,895 | Prediction batches: 46
    Batch 1/46 (1,000/45,895 rows) | 0.7 s
    Batch 2/46 (2,000/45,895 rows) | 1.3 s
    Batch 3/46 (3,000/45,895 rows) | 1.7 s
    Batch 4/46 (4,000/45,895 rows) | 2.3 s
    Batch 5/46 (5,000/45,895 rows) | 2.8 s
    Batch 6/46 (6,000/45,895 rows) | 3.3 s
    Batch 7/46 (7,000/45,895 rows) | 3.8 s
    Batch 8/46 (8,000/45,895 rows) | 4.3 s
    Batch 9/46 (9,000/45,895 rows) | 4.8 s
    Batch 10/46 (10,000/45,895 rows) | 5.4 s
    Batch 11/46 (11,000/45,895 rows) | 5.9 s
    Batch 12/46 (12,000/45,895 rows) | 6.4 s
    Batch 13/46 (13,000/45,895 rows) | 6.8 s
    Batch 14/46 (14,000/45,895 rows) | 7.3 s
    Batch 15/46 (15,000/45,895 rows) | 7.8 s
    Batch 16/46 (16,000/45,895 rows) | 8.3 s
    Batch 17/46 (17,000/45,895 rows) | 8.8 s
    Batch 18/46 (18,000/45,895 rows) | 9.3 s
    Batch 19/46 (19,000/45,895 rows) | 9.8 s
    Batch 20/46 (20,000/45,895 rows) | 10.2 s
    Batch 21/46 (21,000/45,895 rows) | 10.8 s
    Batch 22/46 (22,000/45

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as a different type depending on the ColumnTransformer inputs.
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


  Test rows: 45,895 | Prediction batches: 46
    Batch 1/46 (1,000/45,895 rows) | 0.5 s
    Batch 2/46 (2,000/45,895 rows) | 1.0 s
    Batch 3/46 (3,000/45,895 rows) | 1.5 s
    Batch 4/46 (4,000/45,895 rows) | 2.0 s
    Batch 5/46 (5,000/45,895 rows) | 2.5 s
    Batch 6/46 (6,000/45,895 rows) | 2.9 s
    Batch 7/46 (7,000/45,895 rows) | 3.4 s
    Batch 8/46 (8,000/45,895 rows) | 4.0 s
    Batch 9/46 (9,000/45,895 rows) | 4.4 s
    Batch 10/46 (10,000/45,895 rows) | 4.9 s
    Batch 11/46 (11,000/45,895 rows) | 5.4 s
    Batch 12/46 (12,000/45,895 rows) | 5.9 s
    Batch 13/46 (13,000/45,895 rows) | 6.4 s
    Batch 14/46 (14,000/45,895 rows) | 6.9 s
    Batch 15/46 (15,000/45,895 rows) | 7.4 s
    Batch 16/46 (16,000/45,895 rows) | 7.9 s
    Batch 17/46 (17,000/45,895 rows) | 8.4 s
    Batch 18/46 (18,000/45,895 rows) | 8.9 s
    Batch 19/46 (19,000/45,895 rows) | 9.4 s
    Batch 20/46 (20,000/45,895 rows) | 9.8 s
    Batch 21/46 (21,000/45,895 rows) | 10.4 s
    Batch 22/46 (22,000/45,

...

  Test rows: 13,723 | Prediction batches: 14
    Batch 1/14 (1,000/13,723 rows) | 0.5 s
    Batch 2/14 (2,000/13,723 rows) | 0.9 s
    Batch 3/14 (3,000/13,723 rows) | 1.4 s
    Batch 4/14 (4,000/13,723 rows) | 1.9 s
    Batch 5/14 (5,000/13,723 rows) | 2.4 s
    Batch 6/14 (6,000/13,723 rows) | 2.9 s
    Batch 7/14 (7,000/13,723 rows) | 3.4 s
    Batch 8/14 (8,000/13,723 rows) | 3.9 s
    Batch 9/14 (9,000/13,723 rows) | 4.4 s
    Batch 10/14 (10,000/13,723 rows) | 4.9 s
    Batch 11/14 (11,000/13,723 rows) | 5.4 s
    Batch 12/14 (12,000/13,723 rows) | 5.9 s
    Batch 13/14 (13,000/13,723 rows) | 6.4 s
    Batch 14/14 (13,723/13,723 rows) | 6.9 s
  ROC-AUC: 0.7812 | PR-AUC: 0.7575
  Training time: 0.75 s
  Prediction time: 6.86 s
  Run time: 7.65 s
  Checkpoint saved: 73/110

[74/110] cardiovascular | TabPFN-3 | n=64 | subset_seed=789


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as a different type depending on the ColumnTransformer inputs.
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


  Test rows: 13,723 | Prediction batches: 14
    Batch 1/14 (1,000/13,723 rows) | 0.5 s
    Batch 2/14 (2,000/13,723 rows) | 1.0 s
    Batch 3/14 (3,000/13,723 rows) | 1.4 s
    Batch 4/14 (4,000/13,723 rows) | 2.1 s
    Batch 5/14 (5,000/13,723 rows) | 2.6 s
    Batch 6/14 (6,000/13,723 rows) | 3.0 s
    Batch 7/14 (7,000/13,723 rows) | 3.5 s
    Batch 8/14 (8,000/13,723 rows) | 4.0 s
    Batch 9/14 (9,000/13,723 rows) | 4.5 s
    Batch 10/14 (10,000/13,723 rows) | 5.0 s
    Batch 11/14 (11,000/13,723 rows) | 5.4 s
    Batch 12/14 (12,000/13,723 rows) | 5.9 s
    Batch 13/14 (13,000/13,723 rows) | 6.4 s
    Batch 14/14 (13,723/13,723 rows) | 6.9 s
  ROC-AUC: 0.7126 | PR-AUC: 0.6829
  Training time: 0.74 s
  Prediction time: 6.90 s
  Run time: 7.68 s
  Checkpoint saved: 74/110

[75/110] cardiovascular | TabPFN-3 | n=64 | subset_seed=1011


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as a different type depending on the ColumnTransformer inputs.
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


  Test rows: 13,723 | Prediction batches: 14
    Batch 1/14 (1,000/13,723 rows) | 0.5 s
    Batch 2/14 (2,000/13,723 rows) | 1.0 s
    Batch 3/14 (3,000/13,723 rows) | 1.5 s
    Batch 4/14 (4,000/13,723 rows) | 1.9 s
    Batch 5/14 (5,000/13,723 rows) | 2.4 s
    Batch 6/14 (6,000/13,723 rows) | 2.9 s
    Batch 7/14 (7,000/13,723 rows) | 3.3 s
    Batch 8/14 (8,000/13,723 rows) | 3.8 s
    Batch 9/14 (9,000/13,723 rows) | 4.4 s
    Batch 10/14 (10,000/13,723 rows) | 4.8 s
    Batch 11/14 (11,000/13,723 rows) | 5.4 s
    Batch 12/14 (12,000/13,723 rows) | 5.9 s
    Batch 13/14 (13,000/13,723 rows) | 6.3 s
    Batch 14/14 (13,723/13,723 rows) | 6.9 s
  ROC-AUC: 0.7515 | PR-AUC: 0.7268
  Training time: 0.75 s
  Prediction time: 6.86 s
  Run time: 7.65 s
  Checkpoint saved: 75/110

[76/110] cardiovascular | TabPFN-3 | n=128 | subset_seed=42
  Test rows: 13,723 | Prediction batches: 14
    Batch 1/14 (1,000/13,723 rows) | 0.5 s
    Batch 2/14 (2,000/13,723 rows) | 1.0 s
    Batch 3/14 (3,00

## Finalize and Export TabPFN V3 Results

In [16]:
# ============================================================
# Cell 14 - Finalize and Export TabPFN V3 Results
# ============================================================

PKL_OUTPUT_PATH = Path(
    "/kaggle/working/tabpfn_v3_results_final.pkl"
)

CSV_OUTPUT_PATH = Path(
    "/kaggle/working/tabpfn_v3_results_final.csv"
)

TABPFN_RESULT_COLUMNS = (
    "protocol_version",
    "dataset",
    "model",
    "training_size",
    "subset_seed",
    "model_seed",
    "classification_threshold",
    "training_time",
    "prediction_time",
    *PRIMARY_METRICS,
    *SECONDARY_METRICS,
)

EXPECTED_RUNS = (
    len(EXPECTED_DATASETS)
    * len(TRAINING_SIZES)
    * len(SUBSET_SEEDS)
)

print("Finalizing TabPFN V3 results...")
print("=" * 60)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

results = load_checkpoint()

print(f"Checkpoint loaded: {len(results)} runs")

# ------------------------------------------------------------
# Verify row count
# ------------------------------------------------------------

assert len(results) == EXPECTED_RUNS, (
    f"Expected {EXPECTED_RUNS} runs, "
    f"found {len(results)}."
)

# ------------------------------------------------------------
# Verify result keys and protocol metadata
# ------------------------------------------------------------

for result in results:

    assert set(TABPFN_RESULT_COLUMNS).issubset(
        result.keys()
    )

    assert result["protocol_version"] == PROTOCOL_VERSION
    assert result["model"] == "tabpfn_v3"
    assert result["model_seed"] == MODEL_SEED
    assert (
        result["classification_threshold"]
        == CLASSIFICATION_THRESHOLD
    )

# ------------------------------------------------------------
# Verify unique experimental combinations
# ------------------------------------------------------------

run_keys = {
    (
        r["dataset"],
        r["training_size"],
        r["subset_seed"],
    )
    for r in results
}

assert len(run_keys) == EXPECTED_RUNS, (
    "Duplicate experimental runs detected."
)

expected_keys = {
    (
        dataset_name,
        training_size,
        subset_seed,
    )
    for dataset_name in EXPECTED_DATASETS
    for training_size in TRAINING_SIZES
    for subset_seed in SUBSET_SEEDS
}

assert run_keys == expected_keys, (
    "Results do not contain exactly the expected "
    "dataset/training-size/subset-seed combinations."
)

print("✓ All expected experimental combinations present.")

# ------------------------------------------------------------
# Create deterministic DataFrame
# ------------------------------------------------------------

df = (
    pd.DataFrame(results)
    .loc[:, list(TABPFN_RESULT_COLUMNS)]
    .sort_values(
        by=[
            "dataset",
            "training_size",
            "subset_seed",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save final PKL
# ------------------------------------------------------------

with open(PKL_OUTPUT_PATH, "wb") as handle:
    pickle.dump(
        results,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

# ------------------------------------------------------------
# Save final CSV
# ------------------------------------------------------------

df.to_csv(
    CSV_OUTPUT_PATH,
    index=False,
)

# ------------------------------------------------------------
# Verify output files
# ------------------------------------------------------------

assert PKL_OUTPUT_PATH.exists()
assert CSV_OUTPUT_PATH.exists()

csv_check = pd.read_csv(CSV_OUTPUT_PATH)

assert len(csv_check) == EXPECTED_RUNS

print()
print("=" * 60)
print("TABPFN V3 RESULTS FINALIZED")
print("=" * 60)

print(f"Runs:        {len(df)}")
print(f"PKL:         {PKL_OUTPUT_PATH}")
print(f"CSV:         {CSV_OUTPUT_PATH}")

print()
print("DataFrame shape:")
print(df.shape)

print()
print("Columns:")
print(list(df.columns))

print()
print("✓ Final PKL verified.")
print("✓ Final CSV verified.")
print(f"✓ All {EXPECTED_RUNS} TabPFN V3 runs preserved.")


Finalizing TabPFN V3 results...
Checkpoint loaded: 110 runs
✓ All expected experimental combinations present.

TABPFN V3 RESULTS FINALIZED
Runs:        110
PKL:         /kaggle/working/tabpfn_v3_results_final.pkl
CSV:         /kaggle/working/tabpfn_v3_results_final.csv

DataFrame shape:
(110, 16)

Columns:
['protocol_version', 'dataset', 'model', 'training_size', 'subset_seed', 'model_seed', 'classification_threshold', 'training_time', 'prediction_time', 'roc_auc', 'pr_auc', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1']

✓ Final PKL verified.
✓ Final CSV verified.
✓ All 110 TabPFN V3 runs preserved.


## Final TabPFN V3 Integrity Check

In [18]:
# ============================================================
# Cell 15 - Final Integrity Check — TabPFN V3
# ============================================================

print("Running final TabPFN V3 integrity check...")
print("=" * 60)

print(f"Expected runs: {EXPECTED_RUNS}")
print()

# ------------------------------------------------------------
# Check output files
# ------------------------------------------------------------

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

assert PKL_OUTPUT_PATH.exists(), (
    f"Final PKL not found: {PKL_OUTPUT_PATH}"
)

assert CSV_OUTPUT_PATH.exists(), (
    f"Final CSV not found: {CSV_OUTPUT_PATH}"
)

print("✓ Required output files exist.")
print()

# ------------------------------------------------------------
# Load outputs
# ------------------------------------------------------------

results = load_checkpoint()

with open(PKL_OUTPUT_PATH, "rb") as handle:
    pkl_results = pickle.load(handle)

csv_results = pd.read_csv(CSV_OUTPUT_PATH)

assert pkl_results == results, (
    "Final PKL contents do not match the checkpoint results."
)

pkl_df = (
    pd.DataFrame(pkl_results)
    .loc[:, list(TABPFN_RESULT_COLUMNS)]
)

# ------------------------------------------------------------
# Row counts
# ------------------------------------------------------------

assert len(results) == EXPECTED_RUNS
assert len(pkl_results) == EXPECTED_RUNS
assert len(csv_results) == EXPECTED_RUNS

print("✓ Row counts verified.")

# ------------------------------------------------------------
# Columns
# ------------------------------------------------------------

assert list(pkl_df.columns) == list(
    TABPFN_RESULT_COLUMNS
)

assert list(csv_results.columns) == list(
    TABPFN_RESULT_COLUMNS
)

print("✓ Result columns verified.")

# ------------------------------------------------------------
# Protocol metadata
# ------------------------------------------------------------

assert (
    pkl_df["protocol_version"]
    == PROTOCOL_VERSION
).all()

assert (
    pkl_df["model"]
    == "tabpfn_v3"
).all()

assert (
    pkl_df["model_seed"]
    == MODEL_SEED
).all()

assert (
    pkl_df["classification_threshold"]
    == CLASSIFICATION_THRESHOLD
).all()

print("✓ Protocol metadata verified.")

# ------------------------------------------------------------
# Experimental combinations
# ------------------------------------------------------------

expected_keys = {
    (
        dataset_name,
        training_size,
        subset_seed,
    )
    for dataset_name in EXPECTED_DATASETS
    for training_size in TRAINING_SIZES
    for subset_seed in SUBSET_SEEDS
}

actual_keys = {
    (
        row.dataset,
        row.training_size,
        row.subset_seed,
    )
    for row in pkl_df.itertuples()
}

assert actual_keys == expected_keys, (
    "Experimental combinations do not match protocol."
)

assert len(actual_keys) == EXPECTED_RUNS

print(
    f"✓ All {EXPECTED_RUNS} dataset × size × "
    "subset-seed combinations verified."
)

# ------------------------------------------------------------
# Duplicate check
# ------------------------------------------------------------

duplicate_count = pkl_df.duplicated(
    subset=[
        "dataset",
        "training_size",
        "subset_seed",
    ]
).sum()

assert duplicate_count == 0

print("✓ No duplicate runs detected.")

# ------------------------------------------------------------
# Metric checks
# ------------------------------------------------------------

for metric in ALL_METRICS:

    assert pkl_df[metric].notna().all(), (
        f"NaN values found in {metric}."
    )

    assert np.isfinite(
        pkl_df[metric].astype(float)
    ).all(), (
        f"Non-finite values found in {metric}."
    )

    assert (
        (pkl_df[metric] >= 0)
        & (pkl_df[metric] <= 1)
    ).all(), (
        f"Invalid value outside [0,1] in {metric}."
    )

print("✓ All evaluation metrics are finite and within [0, 1].")

# ------------------------------------------------------------
# Timing checks
# ------------------------------------------------------------

for column in [
    "training_time",
    "prediction_time",
]:

    assert pkl_df[column].notna().all()

    assert (
        pkl_df[column].astype(float) >= 0
    ).all()

print("✓ Training and prediction times verified.")

# ------------------------------------------------------------
# PKL ↔ CSV consistency
# ------------------------------------------------------------

sort_columns = [
    "dataset",
    "training_size",
    "subset_seed",
]

pkl_sorted = (
    pkl_df
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

csv_sorted = (
    csv_results
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    pkl_sorted,
    csv_sorted,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12,
)

print("✓ CSV and PKL contents are consistent.")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("TABPFN V3 FINAL INTEGRITY CHECK PASSED")
print("=" * 60)

print(f"✓ Total runs:       {len(pkl_df)}")
print(f"✓ Datasets:         {pkl_df['dataset'].nunique()}")
print(f"✓ Training sizes:   {pkl_df['training_size'].nunique()}")
print(f"✓ Subset seeds:     {pkl_df['subset_seed'].nunique()}")
print("✓ Model:             tabpfn_v3")
print("✓ No duplicates")
print("✓ No missing metric values")
print("✓ No invalid metric values")
print("✓ Timing values valid")
print("✓ CSV ↔ PKL consistency verified")

print()
print("TabPFN V3 results are ready for analysis.")

Running final TabPFN V3 integrity check...
Expected runs: 110

✓ Required output files exist.

✓ Row counts verified.
✓ Result columns verified.
✓ Protocol metadata verified.
✓ All 110 dataset × size × subset-seed combinations verified.
✓ No duplicate runs detected.
✓ All evaluation metrics are finite and within [0, 1].
✓ Training and prediction times verified.
✓ CSV and PKL contents are consistent.

TABPFN V3 FINAL INTEGRITY CHECK PASSED
✓ Total runs:       110
✓ Datasets:         2
✓ Training sizes:   11
✓ Subset seeds:     5
✓ Model:             tabpfn_v3
✓ No duplicates
✓ No missing metric values
✓ No invalid metric values
✓ Timing values valid
✓ CSV ↔ PKL consistency verified

TabPFN V3 results are ready for analysis.
